# cuda_python — smoke test

`cuda_python` (PyPI `cuda-python`) is a thin **proxy metapackage**: it installs no
module of its own and instead pins its siblings from the same NVIDIA monorepo,
assembling the importable `cuda.*` namespace — `cuda.bindings` (CUDA driver /
runtime / NVRTC bindings), `cuda.core` (Pythonic high-level API), and
`cuda.pathfinder` (the `dlopen` search machinery). This notebook therefore
exercises the `cuda` namespace the metapackage delivers.

In [ ]:
import pkgutil, importlib

# cuda_python is a proxy metapackage: it installs no module of its own and instead
# assembles the `cuda.*` namespace from its siblings (cuda_bindings / cuda_core /
# cuda_pathfinder). We sweep the `cuda` namespace package it delivers.
import cuda as _pkg_under_test

# Minimal denylist -- excluded submodules, each with a documented reason:
_DENY = frozenset({
    # Test-only helpers that hard-require `pytest`, which may not be installed.
    "cuda.bindings._test_helpers.arch_check",
    "cuda.bindings._test_helpers.mempool",
    "cuda.bindings._test_helpers.pep723",
    # Private PyTorch-AOTI interop extension: its `aoti_torch_*` symbols live in
    # libtorch_cpu.so and only resolve through the interop code path (a cold walk
    # import can't make them global) -- not a defect.
    "cuda.core._tensor_bridge",
})

import ctypes as _ctypes

_DRIVER_LOADERS = frozenset({"cuda.tile._load_libcuda"})
try:
    _ctypes.CDLL("libcuda.so.1")
    _driver_error = None
except OSError as _error:
    _driver_error = _error
    print(f"NOTE: the CUDA driver does not load here ({_error}); a driver-loader module that fails only on it is "
          f"accepted: {sorted(_DRIVER_LOADERS)}")

failed = []
for module_info in pkgutil.walk_packages(_pkg_under_test.__path__, prefix=f"{_pkg_under_test.__name__}."):
    if module_info.name in _DENY:
        continue
    try:
        importlib.import_module(module_info.name)
    except Exception as e:
        if (_driver_error is not None and module_info.name in _DRIVER_LOADERS and isinstance(e, OSError)
                and "libcuda.so.1" in str(e)):
            print(f"DRIVER-LOADER SKIP {module_info.name}: {type(e).__name__}: {e}")
            continue
        failed.append((module_info.name, type(e).__name__, str(e)))

if failed:
    for name, etype, msg in failed:
        print(f"FAIL {name}: {etype}: {msg}")
    raise AssertionError(f"{len(failed)} submodule(s) of {_pkg_under_test.__name__} failed to import")
print(f"OK -- all submodules of {_pkg_under_test.__name__} imported cleanly")

## Example 1: metapackage version cohesion

The whole job of the `cuda_python` proxy is to pull its siblings at a matching
version. Assert the installed metapackage version equals `cuda.bindings.__version__`
(they release as the same monorepo tag) and that `cuda.core` / `cuda.pathfinder`
are present.

In [ ]:
import importlib.metadata as md

from packaging.version import Version
import cuda.bindings, cuda.core, cuda.pathfinder

meta_ver = md.version("cuda-python")          # the proxy metapackage itself
b_ver = cuda.bindings.__version__
c_ver = cuda.core.__version__
p_ver = cuda.pathfinder.__version__
print("cuda-python (metapackage):", meta_ver)
print(f"cuda.bindings={b_ver} | cuda.core={c_ver} | cuda.pathfinder={p_ver}")

assert Version(meta_ver).release == Version(b_ver).release, (
    f"proxy {meta_ver} and cuda.bindings {b_ver} are not the same release"
)
assert Version(c_ver).release and Version(p_ver).release, (c_ver, p_ver)   # both siblings present
print("OK -- cuda_python metapackage coheres with its siblings")

## Example 2: cuda.pathfinder locates and loads CUDA libraries

`cuda.pathfinder` is the `dlopen` search machinery. Loading the CUDA runtime and
the NVRTC compiler is **real runtime initialization** and needs no GPU -- it finds
the on-disk libraries from the installed CUDA toolkit and maps them in.

In [ ]:
import os
import cuda.pathfinder as pf

names = pf.SUPPORTED_NVIDIA_LIBNAMES
print(f"pathfinder knows {len(names)} NVIDIA lib names; sample:", sorted(names)[:6])

loaded = {}
for lib in ("cudart", "nvrtc"):
    dl = pf.load_nvidia_dynamic_lib(lib)          # real dlopen, GPU-less
    loaded[lib] = dl.abs_path
    assert os.path.exists(dl.abs_path), f"{lib}: missing path {dl.abs_path}"
    print(f"  loaded {lib:6s} -> {os.path.basename(dl.abs_path)} (found_via={dl.found_via})")

# Idempotent: a repeat load reports the already-loaded handle, same path.
again = pf.load_nvidia_dynamic_lib("cudart")
assert again.abs_path == loaded["cudart"]
print("OK -- pathfinder located + loaded CUDA libraries (no GPU needed)")

## Example 3: driver/runtime version queries + device enumeration

Version queries read the installed toolkit/driver stack and work GPU-less.
Device enumeration is **GPU-conditional**: with no visible GPU, `cudaGetDeviceCount`
returns `cudaErrorNoDevice`, so the enumeration branch prints `SKIP (no GPU)`.
Any other CUDA error fails the cell.

In [ ]:
import cuda.bindings
from cuda.bindings import runtime as rt
from cuda.bindings import driver as drv
from packaging.version import Version

err, rt_ver = rt.cudaRuntimeGetVersion();  assert int(err) == 0, err
err, drv_ver = rt.cudaDriverGetVersion();   assert int(err) == 0, err
print(f"CUDA runtime version: {rt_ver} | driver-API version: {drv_ver}")
# cuda-bindings N.x pairs with the CUDA N.x runtime.
assert rt_ver // 1000 == Version(cuda.bindings.__version__).major, (rt_ver, cuda.bindings.__version__)

d_err, cu_drv_ver = drv.cuDriverGetVersion()   # no cuInit() needed, but libcuda.so.1 IS
assert int(d_err) == 0, d_err
print("driver API reports version:", cu_drv_ver)

# Skip only when no device is visible; any other error fails the cell.
err, count = rt.cudaGetDeviceCount()
if err == rt.cudaError_t.cudaErrorNoDevice or (err == rt.cudaError_t.cudaSuccess and count == 0):
    print(f"SKIP (no GPU) -- cudaGetDeviceCount returned {err}, count={count}")
else:
    assert err == rt.cudaError_t.cudaSuccess, err
    for i in range(count):
        perr, props = rt.cudaGetDeviceProperties(i); assert int(perr) == 0
        raw = props.name
        name = (raw.decode(errors="ignore") if isinstance(raw, (bytes, bytearray)) else str(raw)).split("\x00")[0]
        print(f"  GPU {i}: {name} (SM {props.major}.{props.minor})")
    print(f"OK -- enumerated {count} CUDA device(s)")

## Example 4: NVRTC compiles a CUDA kernel to PTX (GPU-less)

In [ ]:
from cuda.bindings import nvrtc

err, major, minor = nvrtc.nvrtcVersion(); assert int(err) == 0, err
print(f"NVRTC JIT compiler version: {major}.{minor}")

kernel = (b'extern "C" __global__ void saxpy(float a, const float* x,\n'
          b'    const float* y, float* out, int n){\n'
          b'  int i = blockIdx.x*blockDim.x + threadIdx.x;\n'
          b'  if (i < n) out[i] = a*x[i] + y[i];\n}\n')
err, prog = nvrtc.nvrtcCreateProgram(kernel, b"saxpy.cu", 0, [], []); assert int(err) == 0, err
(cerr,) = nvrtc.nvrtcCompileProgram(prog, 1, [b"--gpu-architecture=compute_75"])
assert int(cerr) == 0, f"NVRTC compile failed: {cerr}"

err, ptx_size = nvrtc.nvrtcGetPTXSize(prog); assert int(err) == 0
ptx = bytearray(ptx_size)
(gerr,) = nvrtc.nvrtcGetPTX(prog, ptx); assert int(gerr) == 0 and ptx_size > 0
text = ptx.decode(errors="ignore")
assert ".entry saxpy" in text, "compiled PTX is missing the saxpy kernel entry"
print(f"OK -- NVRTC compiled saxpy to {ptx_size} bytes of PTX (GPU-less)")

## Example 5: cuda.core high-level compile + GPU-conditional launch

`cuda.core.Program` compiles device code via its NVRTC backend with **no GPU**
(the CPU-safe surface asserted below). When a CUDA device is present, the guarded
branch runs the full high-level path: `Device` -> compile to cubin -> `launch` a
kernel -> copy back and numerically verify the result. With no visible device it prints
`SKIP (no GPU)`; any other CUDA error fails the cell.

In [ ]:
import numpy as np, cuda.core as cc
from cuda.bindings import runtime as rt

code = 'extern "C" __global__ void triple(float* y,int n){int i=blockIdx.x*blockDim.x+threadIdx.x; if(i<n) y[i]=3.0f*y[i];}'

prog = cc.Program(code, code_type="c++", options=cc.ProgramOptions(arch="sm_75"))
obj = prog.compile("ptx")                       # CPU-safe: NVRTC backend, no GPU
assert prog.backend == "NVRTC" and isinstance(obj, cc.ObjectCode)
print("cuda.core.Program compiled ->", type(obj).__name__, "| backend:", prog.backend)

# Skip only when no device is visible; any other error fails the cell.
err, count = rt.cudaGetDeviceCount()
if err == rt.cudaError_t.cudaErrorNoDevice or (err == rt.cudaError_t.cudaSuccess and count == 0):
    print(f"SKIP (no GPU) -- cudaGetDeviceCount returned {err}; Device/launch skipped, "
          "cuda.core compile path verified above")
else:
    assert err == rt.cudaError_t.cudaSuccess, err
    dev = cc.Device(0); dev.set_current(); stream = dev.create_stream()
    ccmaj, ccmin = dev.compute_capability
    mod = cc.Program(code, code_type="c++",
                     options=cc.ProgramOptions(arch=f"sm_{ccmaj}{ccmin}")).compile("cubin")
    ker = mod.get_kernel("triple"); n = 4
    pin = cc.LegacyPinnedMemoryResource(); hbuf = pin.allocate(n * 4, stream=stream)
    np.from_dlpack(hbuf).view(np.float32)[:] = [2, 4, 6, 8]
    dbuf = dev.allocate(n * 4, stream=stream); dbuf.copy_from(hbuf, stream=stream)
    cc.launch(stream, cc.LaunchConfig(grid=1, block=n), ker, dbuf, np.int32(n))
    hbuf.copy_from(dbuf, stream=stream); stream.sync()
    out = list(np.from_dlpack(hbuf).view(np.float32))
    assert out == [6, 12, 18, 24], out
    print("OK -- cuda.core kernel launch result:", out)